In [ ]:
dataset_folder="path/2/dataset"
texts_folder="path/2/texts"

In [ ]:
project_folder="path/2/projects"
LANGUAGES=["PT","BG","EN","RU"]

##Get Data

In [ ]:
import json
import os
with open(os.path.join(dataset_folder,"train.json"), "r", newline="") as f:
    train=json.load(f)


with open(os.path.join(dataset_folder,"val.json"), "r", newline="") as f:
    val=json.load(f)
with open(os.path.join(dataset_folder,"test.json"), "r", newline="") as f:
    test=json.load(f)

# Predict Narratives

In [ ]:
def buildPrompt(text,example_text,example_narrative):
  NARRATIVES=[
    "Amplifying Climate Fears",
    "Climate change is beneficial",
    "Controversy about green technologies",
    "Criticism of climate movement",
    "Criticism of climate policies",
    "Criticism of institutions and authorities",
    "Downplaying climate change",
    "Green policies are geopolitical instruments",
    "Hidden plots by secret schemes of powerful groups",
    "Questioning the measurements and science",
    "Amplifying war-related fears",
    "Blaming the war on others rather than the invader",
    "Discrediting Ukraine",
    "Discrediting the West, Diplomacy",
    "Distrust towards Media",
    "Hidden plots by secret schemes of powerful groups",
    "Negative Consequences for the West",
    "Overpraising the West",
    "Praise of Russia",
    "Russia is the Victim",
    "Speculating war outcomes"
  ]

  prompt=f'''

  You are an expert annotator for narrative classification in news articles.

  Your task is to read the article text and assign exactly one dominant narrative category from the provided list of categories related to either the Ukraine-Russia War (URW) or Climate Change (CC).

  A dominant narrative is the category that best captures the article’s main framing, viewpoint, or overall message. Focus on the article as a whole, not on isolated phrases or minor mentions.

  Instructions:
  - Use only the information in the article text.
  - Do not rely on outside knowledge.
  - Do not explain your answer.
  - Do not include evidence, reasoning, or confidence.
  - Choose the most specific applicable category from the provided list.
  - If multiple categories appear, choose the one that best represents the article’s overall purpose.


  Output only the category name, exactly as written in the provided category list.

  Example Article:
  {example_text}

  Example Dominant Narrative:
  {example_narrative}

  Article:
  {text}

  Available categories:
  {NARRATIVES}
  '''
  return prompt

In [ ]:
dataset_folder="path/2/dataset"



In [ ]:
import pandas as pd
import os
import pickle
similarity_map_df=pd.read_csv(os.path.join(dataset_folder,"similarity_map.csv"))


In [ ]:
train_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_train.p"), "rb"))
test_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_test.p"), "rb"))
val_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_val.p"), "rb"))

In [ ]:
similarity_map_df

,Unnamed: 0,language,doc_id,dom_narr,train_most_similar_id,train_most_similar_id_lang,train_most_similar_id_dom_narr,similarity
0,0,PT,PT_318.txt,CC: Amplifying Climate Fears,PT_389.txt,PT,CC: Amplifying Climate Fears,0.727257
1,1,BG,A6_CC_BG_10380.txt,CC: Amplifying Climate Fears,A6_CC_BG_10556.txt,BG,CC: Amplifying Climate Fears,0.777933
2,2,PT,PT_CC_427.txt,CC: Amplifying Climate Fears,PT_401.txt,PT,CC: Criticism of climate policies,0.774226
3,3,BG,A6_CC_BG_8984.txt,CC: Amplifying Climate Fears,BG_364.txt,BG,CC: Amplifying Climate Fears,0.813909
4,4,PT,PT_346.txt,CC: Amplifying Climate Fears,A6_CC_BG_13641.txt,BG,CC: Amplifying Climate Fears,0.845797
...,...,...,...,...,...,...,...,...
113,113,RU,RU-URW-1322.txt,URW: Russia is the Victim,RU-URW-1134.txt,RU,"URW: Discrediting the West, Diplomacy",0.694951
114,114,PT,PT_95.txt,URW: Russia is the Victim,PT_URW_TEST_494.txt,PT,"URW: Discrediting the West, Diplomacy",0.749229
115,115,BG,A4_URW_BG_15138.txt,URW: Speculating war outcomes,A4_URW_BG_15605.txt,BG,URW: Speculating war outcomes,0.681898
116,116,EN,EN_UA_DEV_100002.txt,URW: Speculating war outcomes,EN_UA_101805.txt,EN,URW: Hidden plots by secret schemes of powerfu...,0.762570


In [ ]:

def getExample(doc_id,train_dataset):
  #print(doc_id)
  narrative=[x["dom_narr"] for x in train_dataset if x["doc_id"]==doc_id][0]
  narrative=narrative.split(":")[1]
  text=[x["text"] for x in train_dataset if x["doc_id"]==doc_id][0]
  lang=[x["language"] for x in train_dataset if x["doc_id"]==doc_id][0]

  return narrative, text, lang

In [ ]:
def getDynamicExample(doc_id,train_dataset,similarity_map_df):
  matching=similarity_map_df[similarity_map_df["doc_id"]==doc_id].iloc[0]
  most_similar_id=matching["train_most_similar_id"]
  narrative,text,lang=getExample(most_similar_id,train_dataset)
  return narrative,text,lang


In [ ]:
getDynamicExample(test_embeddings[0]["doc_id"],train_embeddings,similarity_map_df)

In [ ]:
!pip install -U google-generativeai
import google.generativeai as genai
import os



In [ ]:
def promptGEMINI(prompt,model_name="gemini-2.5-flash-lite"):
  genai.configure(api_key=LANGEXTRACT_API_KEY)

  model = genai.GenerativeModel(
      model_name=model_name
  )


  response = model.generate_content(prompt)

  # 4. Print the results
  print("-" * 30)
  print(f"Response:\n{response.text}")
  print("-" * 30)
  return response

In [ ]:
LANGEXTRACT_API_KEY="langextract_key"
#gemma-4-31b-it
#gemma-4-26b-a4b-it
def extract_info_gemini_models(input_text,example_text,example_narrative,model="gemini-2.5-flash-lite"):

  prompt=buildPrompt(input_text,example_narrative=example_narrative,example_text=example_text)
  #query gemini
  result=promptGEMINI(prompt,model)
  return result.text


In [ ]:
input_text="RU-URW-1183.txt"
example_narrative,example_text,lang=getDynamicExample(input_text,train_embeddings,similarity_map_df)
res=extract_info_gemini_models(input_text,example_text,example_narrative,model="gemini-2.5-flash-lite")

------------------------------
Response:
Negative Consequences for the West
------------------------------


In [ ]:
res

'Negative Consequences for the West'

##Results

In [ ]:
#gemma-4-31b-it
#gemma-4-26b-a4b-it
#gpt4o
import os
import json
from tqdm import tqdm
import time
output_folder="path/2/results"
def process_test_set(train,test,experiment_type,model,similarity_map):
  experiment_folder=os.path.join(output_folder,experiment_type,model)
  if(not os.path.exists(experiment_folder)):
    os.makedirs(os.path.join(output_folder,experiment_type,model))
    print("folder created")
  for entry in tqdm(test):

    file_check=os.path.join(experiment_folder,str(entry["doc_id"])+".json")
    print(file_check)
    if(os.path.exists(file_check)):
      print("file already exists")
      continue


    input_text=entry["text"]
    print(input_text)
    example_narrative,example_text,lang=getDynamicExample(entry["doc_id"],train_embeddings,similarity_map_df)
    print(example_narrative,example_text,lang)
    print("RES TIME")
    res=extract_info_gemini_models(input_text,example_text,example_narrative,model="gemini-2.5-flash-lite")
    print(res)
    print("processed!!!",entry["doc_id"])

    #build json
    temp=dict()
    temp["doc_id"]=entry["doc_id"]
    temp["predicted_dom_narrative"]=res.strip()

    with open(os.path.join(experiment_folder,str(entry["doc_id"])+".json"), "w") as f:
      json.dump(temp,f)


    print("written!!!",entry["doc_id"])
    time.sleep(90)



In [ ]:
process_test_set(train=train_embeddings,
                 test=test_embeddings,
                 experiment_type="narrative_prediction",
                 model="gemini-2.5-flash-lite",
                 similarity_map=similarity_map_df)

In [ ]:
entry


#Extract Evidence based on Predicted Narratives


In [ ]:
!pip install langextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.1/142.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 5.2 MB/s eta 0:00:00


In [ ]:
dataset_folder="path/2/dataset"
texts_folder="path/2/raw_texts"

In [ ]:
project_folder="/content/drive/MyDrive/LIAAD/evidence_paper/projects"
LANGUAGES=["PT","BG","EN","RU"]

In [ ]:
import os
output_folder="/content/drive/MyDrive/LIAAD/evidence_paper/results"
narrative_prediction_folder= os.path.join(output_folder,"narrative_prediction","gemini-2.5-flash-lite")


In [ ]:
def loadPredictedNarrative(narrative_prediction_folder,doc_id):
  with open(os.path.join(narrative_prediction_folder,str(doc_id)+".json"), "r") as f:
    temp=json.load(f)
  return temp["predicted_dom_narrative"]


In [ ]:
for entry in test:
  try:
    loadPredictedNarrative(narrative_prediction_folder,entry["doc_id"])
  except Exception as e:
    print(entry["doc_id"],"FAILED",str(e))
    continue


##Get Evidence

In [ ]:
!pip install langextract

In [ ]:
import pandas as pd
import os
import pickle
similarity_map_df=pd.read_csv(os.path.join(dataset_folder,"similarity_map.csv"))


In [ ]:
train_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_train.p"), "rb"))
test_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_test.p"), "rb"))
val_embeddings=pickle.load(open(os.path.join(dataset_folder,"embeddings_val.p"), "rb"))

In [ ]:
similarity_map_df

,Unnamed: 0,language,doc_id,dom_narr,train_most_similar_id,train_most_similar_id_lang,train_most_similar_id_dom_narr,similarity
0,0,PT,PT_318.txt,CC: Amplifying Climate Fears,PT_389.txt,PT,CC: Amplifying Climate Fears,0.727257
1,1,BG,A6_CC_BG_10380.txt,CC: Amplifying Climate Fears,A6_CC_BG_10556.txt,BG,CC: Amplifying Climate Fears,0.777933
2,2,PT,PT_CC_427.txt,CC: Amplifying Climate Fears,PT_401.txt,PT,CC: Criticism of climate policies,0.774226
3,3,BG,A6_CC_BG_8984.txt,CC: Amplifying Climate Fears,BG_364.txt,BG,CC: Amplifying Climate Fears,0.813909
4,4,PT,PT_346.txt,CC: Amplifying Climate Fears,A6_CC_BG_13641.txt,BG,CC: Amplifying Climate Fears,0.845797
...,...,...,...,...,...,...,...,...
113,113,RU,RU-URW-1322.txt,URW: Russia is the Victim,RU-URW-1134.txt,RU,"URW: Discrediting the West, Diplomacy",0.694951
114,114,PT,PT_95.txt,URW: Russia is the Victim,PT_URW_TEST_494.txt,PT,"URW: Discrediting the West, Diplomacy",0.749229
115,115,BG,A4_URW_BG_15138.txt,URW: Speculating war outcomes,A4_URW_BG_15605.txt,BG,URW: Speculating war outcomes,0.681898
116,116,EN,EN_UA_DEV_100002.txt,URW: Speculating war outcomes,EN_UA_101805.txt,EN,URW: Hidden plots by secret schemes of powerfu...,0.762570


In [ ]:
import langextract as lx

def getEvidenceFromPosition(text,list_positions):
  evidences=list()
  for entry in list_positions:
    begin=entry["begin"]
    end=entry["end"]
    evidence=text[begin:end]
    evidences.append(evidence)
  return evidences


def getExamples(doc_id,train_dataset):
  #print(doc_id)
  extractions=[x["evidence"] for x in train_dataset if x["doc_id"]==doc_id][0]
  text=[x["text"] for x in train_dataset if x["doc_id"]==doc_id][0]
  #print(extractions)
  extractions=getEvidenceFromPosition(text,extractions)

  #print(extractions)
  lx_extract=list()
  for ex in extractions:
    lx_extract.append(lx.data.Extraction(extraction_class="evidence", extraction_text=ex),)

  narrative=[x["dom_narr"] for x in train_dataset if x["doc_id"]==doc_id][0]
  lang=[x["language"] for x in train_dataset if x["doc_id"]==doc_id][0]

  #print(narrative,lang)
  # Define example data with entities in order of appearance
  examples = [
      lx.data.ExampleData(
          text="NARRATIVE: " + narrative + "\n\n TEXT:" + text,
          extractions=lx_extract
      )
  ]
  return examples

In [ ]:
def getDynamicExample(doc_id,train_dataset,similarity_map_df):
  matching=similarity_map_df[similarity_map_df["doc_id"]==doc_id].iloc[0]
  most_similar_id=matching["train_most_similar_id"]
  examples=getExamples(most_similar_id,train_dataset)
  return examples


In [ ]:
getDynamicExample(test_embeddings[0]["doc_id"],train_embeddings,similarity_map_df)

In [ ]:
import langextract as lx
GOOGLE_API_KEY="google-api-key"

def extract_info_gemini_models(example_annotations,input_text,predicted_input_narrative,model="gemini-2.5-flash-lite"):

  # Define extraction prompt
  prompt_description = "Extract all text spans that support and justify the selected dominant narrative: \n \n  DOMINANT NARRATIVE:"+ predicted_input_narrative

  result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt_description,
    examples=example_annotations,
    api_key=LANGEXTRACT_API_KEY,
    model_id=model
)
  return result


In [ ]:
#gemma-4-31b-it
#gemma-4-26b-a4b-it
#gpt4o
import os
from tqdm import tqdm
import time
output_folder="path/2/results"
def process_test_set_predicted(train,test,experiment_type,model,similarity_map):
  experiment_folder=os.path.join(output_folder,experiment_type,model)
  if(not experiment_folder):
    os.makedirs(os.path.join(output_folder,model))
    print("folder created")
  for entry in tqdm(test):

    file_check=os.path.join(experiment_folder,str(entry["doc_id"])+".json")
    print(file_check)
    if(os.path.exists(file_check)):
      print("file already exists")
      continue


    examples= getDynamicExample(entry["doc_id"],similarity_map_df=similarity_map,train_dataset=train)
    input_text=entry["text"]
    predicted_input_narrative=loadPredictedNarrative(narrative_prediction_folder,entry["doc_id"])
    print("RES TIME")

    res=extract_info_gemini_models(examples,input_text,predicted_input_narrative,model)

    print("processed!!!",entry["doc_id"])

    lx.io.save_annotated_documents([res], output_name=str(entry["doc_id"])+".json", output_dir=experiment_folder)
    html_content = lx.visualize(os.path.join(experiment_folder,str(entry["doc_id"])+".json"))

    with open(os.path.join(experiment_folder,str(entry["doc_id"])+".html"), "w") as f:
        if hasattr(html_content, 'data'):
            f.write(html_content.data)  # For Jupyter/Colab
        else:
            f.write(html_content)

    print("written!!!",entry["doc_id"])
    time.sleep(10)



In [ ]:
process_test_set_predicted(train=train_embeddings,
                 test=test_embeddings,
                 experiment_type="evidence_based_predicted_narrative",
                 model="gemini-2.5-flash-lite",
                 similarity_map=similarity_map_df)

#Debugging

In [ ]:
#For Debug Can be useful for further testing
for ind,row in test.iterrows():
  try:
    examples=getDynamicExample(row["id"],lang=row["language"],similarity_map_df=similarity_map_df,train_dataset=train)
  except Exception as e:
    print(row["id"],"FAILED",str(e))
    continue
  else:
    print(row["id"],"Success")


In [ ]:
any(similarity_map_df["id"]=="RU-URW-1139.txt")